# 05 — EDA: análisis exploratorio de la producción 2024–2026

**Objetivo:** responder cuatro preguntas de negocio con gráficos, a partir del archivo limpio de producción.

| Dato | Valor |
| --- | --- |
| Archivo que se lee | `data/clean/produccion_2024-2026_limpio.parquet` (solo `data/clean`; `data/raw` no se toca) |
| Generado por | `04_produccion_limpieza.ipynb` (ver `data/clean/README.md`) |
| Período | Enero 2024 a agosto 2026 |
| Unidades | Petróleo en **m³**, gas en **miles de m³** (las de la fuente: no se convierte nada) |
| Gráficos | Se guardan como PNG en `reports/figures/` |

**Decisiones de alcance** (aprobadas el 14/10/2026):
- **Petróleo y gas van siempre en paneles separados:** tienen unidades distintas y no se suman.
- **Gráfico 2:** el ranking es de **2025** (último año completo); para comparar los tres años se usa **enero a agosto** de cada año, así los tres tienen los mismos 8 meses.
- **Gráfico 3:** solo **2025**.
- **Gráfico 4:** solo pozos que arrancaron en **febrero de 2024 o después**; se muestran el **promedio y la mediana**; la curva se corta cuando quedan **menos de 30 pozos**.

**Una aclaración sobre "pozo":** en estos datos, `idpozo` identifica un **pozo + formación** (ver el diccionario de datos).
Un pozo físico que produce de dos formaciones cuenta dos veces. En los gráficos 3 y 4, "pozo" significa `idpozo`.

## 1. Preparar el entorno

- **pandas** trabaja con tablas y **numpy** hace cálculos numéricos (lo usamos para los rangos logarítmicos del gráfico 3).
- **matplotlib** dibuja los gráficos. `FuncFormatter` permite escribir los números de los ejes con punto de miles (2.500.000).
- **os** crea la carpeta `reports/figures/` si no existe.

Definimos también un **estilo común** para los cuatro gráficos: colores, tamaño de letra y grilla suave.
Los colores salen de una paleta probada para personas con daltonismo (azul y naranja se distinguen bien entre sí).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter, MaxNLocator

RUTA_CLEAN = "../data/clean/produccion_2024-2026_limpio.parquet"
CARPETA_FIGURAS = "../reports/figures"
os.makedirs(CARPETA_FIGURAS, exist_ok=True)

print("Versión de pandas:    ", pd.__version__)
print("Versión de matplotlib:", plt.matplotlib.__version__)

# Estilo común
AZUL, NARANJA, GRIS = "#2a78d6", "#eb6834", "#8a8984"
TINTA, TINTA_SUAVE = "#0b0b0b", "#52514e"
AZULES_AÑO = {2024: "#86b6ef", 2025: "#2a78d6", 2026: "#104281"}  # del más claro al más oscuro

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c9c8c3", "axes.labelcolor": TINTA_SUAVE, "axes.titlecolor": TINTA,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 10,
    "xtick.color": TINTA_SUAVE, "ytick.color": TINTA_SUAVE,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "font.size": 10,
})

# Números con punto de miles, como se escriben en Argentina: 2.500.000
miles = FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", "."))

# Meses en castellano para el eje de tiempo: "ene 2024", "jul 2024"...
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
mes_anio = FuncFormatter(lambda x, _: f"{MESES[mdates.num2date(x).month - 1]} {mdates.num2date(x).year}")


def guardar(fig, nombre):
    ruta = os.path.join(CARPETA_FIGURAS, nombre)
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    print("Guardado:", ruta)

## 2. Cargar el archivo limpio

Leemos el Parquet. Al venir de Parquet, `fecha_mes` ya es una fecha y las marcas ya son sí/no: no hay que convertir nada.
Mostramos las filas y el período para confirmar que es el archivo esperado (2.628.689 filas según `data/clean/README.md`).

In [ ]:
df = pd.read_parquet(RUTA_CLEAN)
print(f"Filas: {len(df):,}  |  Columnas: {df.shape[1]}")
print("Período:", df["fecha_mes"].min().date(), "a", df["fecha_mes"].max().date())

## Gráfico 1 · Producción mensual por tipo de recurso

**Pregunta de negocio:** ¿cómo evolucionó la producción total de petróleo y de gas entre 2024 y 2026,
y cuánto aporta el recurso **no convencional** (Vaca Muerta y similares) frente al **convencional**?

**Cómo se calcula:**
1. Agrupamos por mes (`fecha_mes`) y tipo de recurso, y sumamos la producción de todos los pozos (`groupby` + `sum`).
2. `unstack` pone cada tipo de recurso en su propia columna: una línea por tipo.
3. Dibujamos dos paneles, uno para petróleo y otro para gas.

Solo se grafican CONVENCIONAL y NO CONVENCIONAL. "NO DISCRIMINADO" y "SIN RESERVORIO" quedan afuera porque son mínimos;
la celda siguiente informa cuánto suman, para que la exclusión quede documentada.

In [ ]:
TIPOS = ["CONVENCIONAL", "NO CONVENCIONAL"]
COLOR_TIPO = {"CONVENCIONAL": AZUL, "NO CONVENCIONAL": NARANJA}

excluidos = df[~df["tipo_de_recurso"].isin(TIPOS)]
print("Filas excluidas por tipo de recurso:", f"{len(excluidos):,}")
print(f"  petróleo: {excluidos['prod_pet_m3'].sum():,.0f} m³ ({excluidos['prod_pet_m3'].sum() / df['prod_pet_m3'].sum() * 100:.2f}% del total)")
print(f"  gas:      {excluidos['prod_gas_miles_m3'].sum():,.0f} miles de m³ ({excluidos['prod_gas_miles_m3'].sum() / df['prod_gas_miles_m3'].sum() * 100:.2f}% del total)")

mensual = (df[df["tipo_de_recurso"].isin(TIPOS)]
           .groupby(["fecha_mes", "tipo_de_recurso"])[["prod_pet_m3", "prod_gas_miles_m3"]]
           .sum()
           .unstack("tipo_de_recurso"))
mensual["prod_pet_m3"].tail(3)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)
paneles = [
    ("prod_pet_m3", "Petróleo", "Producción de petróleo (m³)"),
    ("prod_gas_miles_m3", "Gas", "Producción de gas (miles de m³)"),
]
for eje, (columna, titulo, etiqueta_y) in zip(ejes, paneles):
    for tipo in TIPOS:
        serie = mensual[columna][tipo]
        eje.plot(serie.index, serie.values, color=COLOR_TIPO[tipo], linewidth=2, label=tipo.capitalize())
        eje.annotate(tipo.capitalize(), xy=(serie.index[-1], serie.values[-1]), xytext=(6, 0),
                     textcoords="offset points", va="center", fontsize=9, color=TINTA_SUAVE)
    eje.set_title(titulo, loc="left")
    eje.set_xlabel("Mes")
    eje.set_ylabel(etiqueta_y)
    eje.yaxis.set_major_formatter(miles)
    eje.set_ylim(bottom=0)
    eje.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
    eje.xaxis.set_major_formatter(mes_anio)
    eje.legend(loc="lower left")

fig.suptitle("Producción mensual por tipo de recurso, enero 2024 – agosto 2026", x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.text(0.01, -0.03, "Fuente: Secretaría de Energía, Capítulo IV. Elaboración propia (notebook 05).", fontsize=8, color=TINTA_SUAVE)
fig.tight_layout()
guardar(fig, "01_produccion_mensual_por_recurso.png")
plt.show()

**Observación:**

## Gráfico 2 · Top 10 empresas por producción en 2025

**Pregunta de negocio:** ¿qué empresas concentran la producción del último año completo (2025),
y cómo cambió su producción entre 2024 y 2026?

**Cómo se calcula:**
1. **Ranking:** sumamos la producción de **todo 2025** por empresa y nos quedamos con las 10 mayores (`nlargest`).
   Petróleo y gas tienen rankings distintos (una empresa puede ser grande en gas y chica en petróleo).
2. **Comparación entre años:** para esas 10 empresas sumamos **enero a agosto** de 2024, 2025 y 2026.
   Se usa el mismo tramo de 8 meses porque 2026 llega hasta agosto: comparar años completos con un año incompleto engañaría.
3. Barras horizontales: una barra por año, del más claro (2024) al más oscuro (2026).

**Recordatorio (D-16):** cada mes cuenta para la empresa que lo operó. Si una empresa compró áreas en 2024–2025,
su barra crece y la de la vendedora baja: es un cambio real de operador, no un error.

In [ ]:
TOP = 10
anio_2025 = df[df["anio"] == 2025]
ene_ago = df[df["mes"] <= 8]


def top_empresas(columna):
    ranking = anio_2025.groupby("empresa")[columna].sum().nlargest(TOP)
    por_anio = (ene_ago[ene_ago["empresa"].isin(ranking.index)]
                .groupby(["empresa", "anio"])[columna].sum()
                .unstack("anio")
                .reindex(ranking.index))
    return ranking, por_anio


ranking_pet, ene_ago_pet = top_empresas("prod_pet_m3")
ranking_gas, ene_ago_gas = top_empresas("prod_gas_miles_m3")
print("Top 10 petróleo 2025 (m³):")
print(ranking_pet.map("{:,.0f}".format).to_string())
print("\nTop 10 gas 2025 (miles de m³):")
print(ranking_gas.map("{:,.0f}".format).to_string())

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(15, 7))
paneles = [
    (ene_ago_pet, "Petróleo", "Producción de petróleo, enero a agosto (m³)"),
    (ene_ago_gas, "Gas", "Producción de gas, enero a agosto (miles de m³)"),
]
alto = 0.26
for eje, (tabla, titulo, etiqueta_x) in zip(ejes, paneles):
    posiciones = np.arange(len(tabla))
    for i, anio in enumerate([2024, 2025, 2026]):
        eje.barh(posiciones + (i - 1) * alto, tabla[anio].fillna(0), height=alto - 0.03,
                 color=AZULES_AÑO[anio], label=str(anio))
    eje.set_yticks(posiciones, tabla.index)
    eje.invert_yaxis()  # la empresa más grande arriba
    eje.set_title(titulo, loc="left")
    eje.set_xlabel(etiqueta_x)
    eje.xaxis.set_major_locator(MaxNLocator(nbins=4))
    eje.xaxis.set_major_formatter(miles)
    eje.grid(axis="y", visible=False)
    eje.legend(title="Año (ene–ago)", loc="lower right")

fig.suptitle("Top 10 empresas por producción en 2025, comparadas en enero–agosto de 2024, 2025 y 2026",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.text(0.01, -0.02, "Ranking por producción de todo 2025. Fuente: Secretaría de Energía, Capítulo IV. Elaboración propia (notebook 05).",
         fontsize=8, color=TINTA_SUAVE)
fig.tight_layout()
guardar(fig, "02_top10_empresas_2025.png")
plt.show()

**Observación:**

## Gráfico 3 · Distribución de la producción de petróleo por pozo (2025)

**Pregunta de negocio:** ¿la producción está repartida entre muchos pozos parecidos, o concentrada en unos pocos muy productivos?

**Cómo se calcula:**
1. Sumamos el petróleo de **todo 2025** de cada pozo (`idpozo`).
2. Dejamos afuera los pozos con **0 o menos**: el logaritmo de 0 no existe y no se pueden ubicar en un eje logarítmico.
   No son errores (son pozos de gas, inyectores o parados: *cero no es nulo*); la celda informa cuántos son.
3. Dibujamos un **histograma**: cada barra cuenta cuántos pozos producen dentro de un rango.

**¿Por qué escala logarítmica?** Hay pozos que producen menos de 1 m³ al año y otros más de 100.000 m³.
En una escala común, casi todos quedarían apretados contra el cero. En la escala logarítmica, cada marca del eje vale
**10 veces** la anterior (1, 10, 100, 1.000...), y los rangos de las barras también crecen de a múltiplos (`np.logspace`).

In [ ]:
pet_por_pozo = anio_2025.groupby("idpozo")["prod_pet_m3"].sum()
con_petroleo = pet_por_pozo[pet_por_pozo > 0]

print(f"Pozos con datos en 2025:          {len(pet_por_pozo):,}")
print(f"  con petróleo > 0 (se grafican): {len(con_petroleo):,}")
print(f"  con petróleo = 0 (quedan fuera): {(pet_por_pozo == 0).sum():,}")
print(f"  con petróleo < 0 (quedan fuera): {(pet_por_pozo < 0).sum():,}")
print(f"\nMediana: {con_petroleo.median():,.0f} m³  |  10% más productivo: más de {con_petroleo.quantile(0.9):,.0f} m³")
top10_pct = con_petroleo.nlargest(int(len(con_petroleo) * 0.1)).sum() / con_petroleo.sum() * 100
print(f"El 10% de pozos más productivo aporta el {top10_pct:.0f}% del petróleo de 2025")

In [ ]:
rangos = np.logspace(np.floor(np.log10(con_petroleo.min())), np.ceil(np.log10(con_petroleo.max())), 50)

fig, eje = plt.subplots(figsize=(11, 5))
eje.hist(con_petroleo, bins=rangos, color=AZUL, edgecolor="#fcfcfb", linewidth=0.6)
eje.set_xscale("log")
eje.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", ".") if x >= 1 else f"{x:g}".replace(".", ",")))

mediana = con_petroleo.median()
eje.axvline(mediana, color=TINTA, linestyle="--", linewidth=1.2)
eje.annotate(f"Mediana: {mediana:,.0f} m³".replace(",", "."), xy=(mediana, eje.get_ylim()[1] * 0.97),
             xytext=(-8, 0), textcoords="offset points", ha="right", fontsize=9, color=TINTA,
             bbox={"facecolor": "#fcfcfb", "edgecolor": "none", "pad": 1})

eje.set_title("Producción de petróleo por pozo en 2025", loc="left")
eje.set_xlabel("Petróleo producido en 2025 por pozo (m³, escala logarítmica)")
eje.set_ylabel("Cantidad de pozos")
eje.yaxis.set_major_formatter(miles)
eje.grid(axis="x", visible=False)
fig.text(0.01, -0.04, f"{len(con_petroleo):,} pozos (idpozo) con petróleo > 0 en 2025. ".replace(",", ".")
         + "Fuente: Secretaría de Energía, Capítulo IV. Elaboración propia (notebook 05).", fontsize=8, color=TINTA_SUAVE)
fig.tight_layout()
guardar(fig, "03_histograma_petroleo_por_pozo_2025.png")
plt.show()

**Observación:**

## Gráfico 4 · Curva de declive promedio de los pozos no convencionales de Vaca Muerta

**Pregunta de negocio:** ¿cómo evoluciona la producción de petróleo de un pozo típico de Vaca Muerta desde que empieza a producir?
¿En qué mes llega a su pico y qué tan rápido cae después?

### La idea, con un ejemplo de 3 pozos

Cada pozo arranca en un mes distinto del calendario. Si los graficáramos por fecha, no se podrían comparar.
La solución es **alinearlos en el mismo punto de partida**, como si todos hubieran arrancado el mismo día:

| Pozo | Primer mes con petróleo | Producción por mes del calendario |
| --- | --- | --- |
| A | marzo 2024 | mar: 100 · abr: 400 · may: 300 |
| B | junio 2024 | jun: 200 · jul: 600 · ago: 500 |
| C | enero 2025 | ene: 300 · feb: 500 · mar: 400 |

**Paso 1:** para cada pozo, su primer mes con petróleo mayor a 0 pasa a ser su **"mes 0"**.

**Paso 2:** cada fila pasa a medirse en **meses desde el inicio**: para el pozo A, marzo = mes 0, abril = mes 1, mayo = mes 2.

| Meses desde el inicio | Pozo A | Pozo B | Pozo C | **Promedio** | **Mediana** |
| --- | --- | --- | --- | --- | --- |
| 0 | 100 | 200 | 300 | **200** | **200** |
| 1 | 400 | 600 | 500 | **500** | **500** |
| 2 | 300 | 500 | 400 | **400** | **400** |

**Paso 3:** para cada "mes desde el inicio" calculamos el **promedio** y la **mediana** entre todos los pozos. Esa es la curva.

### Tres cuidados (decisiones aprobadas el 14/10/2026)

1. **Solo pozos que arrancaron en febrero de 2024 o después.** Los datos empiezan en enero de 2024: si un pozo ya producía ese mes,
   no sabemos si era su primer mes o si venía de antes. Incluirlo mezclaría un pozo viejo con uno nuevo.
2. **Promedio y mediana.** La mediana muestra al pozo "del medio"; el promedio se infla con unos pocos pozos excepcionales.
   Si las dos líneas se separan mucho, hay pozos muy distintos al resto.
3. **La curva se corta cuando quedan menos de 30 pozos.** Un pozo que arrancó en 2026 todavía no tiene "mes 20".
   Cuanto más lejos del inicio, menos pozos hay para promediar, y con muy pocos la curva salta y engaña.
   El panel de abajo muestra cuántos pozos hay en cada mes.

**Ojo con el mes 0:** suele ser bajo porque el pozo empieza a producir **a mitad de mes**, no porque produzca poco.

**Filtro de Vaca Muerta no convencional (D-17):** `formprod == "VMUT"` y `tipo_de_recurso == "NO CONVENCIONAL"`.

In [ ]:
MIN_POZOS = 30
INICIO_MINIMO = pd.Timestamp("2024-02-01")

vm = df[(df["formprod"] == "VMUT") & (df["tipo_de_recurso"] == "NO CONVENCIONAL")]

# Paso 1: primer mes con petróleo > 0 de cada pozo
inicio = vm[vm["prod_pet_m3"] > 0].groupby("idpozo")["fecha_mes"].min()
nuevos = inicio[inicio >= INICIO_MINIMO]

print(f"Pozos de Vaca Muerta no convencional:               {vm['idpozo'].nunique():,}")
print(f"  con petróleo en el período:                        {len(inicio):,}")
print(f"  que ya producían en enero de 2024 (quedan fuera):  {(inicio < INICIO_MINIMO).sum():,}")
print(f"  que arrancaron en febrero de 2024 o después:       {len(nuevos):,}")

**Paso 2 en código:** a cada fila le agregamos el mes de inicio de su pozo (`map`) y calculamos la diferencia en meses:
(años de diferencia × 12) + (meses de diferencia). Las filas anteriores al inicio (meses sin petróleo antes de arrancar) quedan afuera.

**Paso 3 en código:** agrupamos por "meses desde el inicio" y calculamos cantidad de pozos, promedio y mediana (`agg`).

In [ ]:
curva = vm[vm["idpozo"].isin(nuevos.index)].copy()
curva["inicio"] = curva["idpozo"].map(nuevos)
curva["meses_desde_inicio"] = ((curva["fecha_mes"].dt.year - curva["inicio"].dt.year) * 12
                               + (curva["fecha_mes"].dt.month - curva["inicio"].dt.month))
curva = curva[curva["meses_desde_inicio"] >= 0]

declive = curva.groupby("meses_desde_inicio")["prod_pet_m3"].agg(pozos="count", promedio="mean", mediana="median")
declive_graficado = declive[declive["pozos"] >= MIN_POZOS]

pico = declive_graficado["mediana"].idxmax()
print(f"Meses graficados: 0 a {declive_graficado.index.max()} (después quedan menos de {MIN_POZOS} pozos)")
print(f"Pico de la mediana: mes {pico}, con {declive_graficado.loc[pico, 'mediana']:,.0f} m³")
ultimo = declive_graficado.index.max()
print(f"En el mes {ultimo}, la mediana es el {declive_graficado.loc[ultimo, 'mediana'] / declive_graficado.loc[pico, 'mediana'] * 100:.0f}% del pico")
declive_graficado.round(0).head(6)

In [ ]:
fig, (eje, eje_n) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

x = declive_graficado.index
eje.plot(x, declive_graficado["promedio"], color=AZUL, linewidth=2, marker="o", markersize=4, label="Promedio")
eje.plot(x, declive_graficado["mediana"], color=NARANJA, linewidth=2, marker="o", markersize=4, label="Mediana")
for nombre, color in [("promedio", AZUL), ("mediana", NARANJA)]:
    eje.annotate(nombre.capitalize(), xy=(x[-1], declive_graficado[nombre].iloc[-1]), xytext=(6, 0),
                 textcoords="offset points", va="center", fontsize=9, color=TINTA_SUAVE)
eje.set_title("Curva de declive: pozos no convencionales de Vaca Muerta", loc="left")
eje.set_ylabel("Petróleo por pozo en el mes (m³)")
eje.yaxis.set_major_formatter(miles)
eje.set_ylim(bottom=0)
eje.legend(loc="upper right")

eje_n.bar(x, declive_graficado["pozos"], color=GRIS, width=0.7)
eje_n.set_ylabel("Pozos")
eje_n.set_xlabel("Meses desde el primer mes con producción de petróleo")
eje_n.yaxis.set_major_formatter(miles)
eje_n.grid(axis="x", visible=False)

fig.text(0.01, -0.02, f"{len(nuevos):,} pozos (idpozo) que arrancaron entre febrero 2024 y agosto 2026; ".replace(",", ".")
         + f"curva cortada con menos de {MIN_POZOS} pozos. Fuente: Secretaría de Energía, Capítulo IV. Elaboración propia (notebook 05).",
         fontsize=8, color=TINTA_SUAVE)
fig.tight_layout()
guardar(fig, "04_declive_vaca_muerta.png")
plt.show()

**Observación:**

## Hallazgos de calidad

Problemas de calidad **nuevos** que aparecieron durante el EDA. Según la regla del proyecto, **no se corrigen acá**:
se registran para decidir qué hacer en una próxima versión de la limpieza. La evidencia está en la celda de abajo.

| Código | Dimensión | Hallazgo | Impacto en el EDA |
| --- | --- | --- | --- |
| H-45 | Consistencia | **Un mismo grupo empresario figura con varias razones sociales**, cada una con su propio código: Pluspetrol (3), CGC (2), GeoPark (2), Capetrol (2). | El gráfico 2 las trata como empresas distintas: el peso de esos grupos queda repartido. Puede ser correcto (son sociedades distintas), pero conviene decidir si el análisis es por sociedad o por grupo |
| H-46 | Consistencia | **Nombres de empresa con formato distinto**: algunos en minúsculas (`Clear Petroleum S.A.`, `Pilgrim Energy S.A.`) y otros con abreviaturas sin unificar (`S A`, `SAU.`, `S.A`) | Solo estético en el EDA; afecta si en el futuro se cruzan empresas por nombre en lugar de por código |
| H-47 | Exactitud | **Pozos con menos de 1 m³ de petróleo en todo 2025** (339 pozos) | Aparecen en el extremo izquierdo del gráfico 3. Pueden ser restos de producción o redondeos; a investigar |

In [ ]:
print("H-45 · Razones sociales del mismo grupo (código | nombre):")
grupos = df["empresa"].str.contains("PLUSPETROL|COMBUSTIBLES|CGC|GEOPARK|CAPETROL", case=False)
print(df.loc[grupos, ["idempresa", "empresa"]].drop_duplicates().sort_values("empresa").to_string(index=False))

print("\nH-46 · Nombres de empresa que no están en mayúsculas:")
nombres = pd.Series(df["empresa"].unique())
print(nombres[nombres != nombres.str.upper()].to_string(index=False))

print(f"\nH-47 · Pozos con 0 < petróleo 2025 < 1 m³: {((pet_por_pozo > 0) & (pet_por_pozo < 1)).sum():,}")